## Домашнее задание – 5 (Использование TAM)

### Задача:
Проверить как отработает на вашей задаче и корпусе метод TAM (https://github.com/xmed-lab/TAM). Если все плохо, то в выводах так и написать. На паре мы как раз обсуждали, что интересно, но не факт что будет отлично работать как пишут.

In [1]:
import os
import torch
import numpy as np
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

from TAM.qwen_utils import process_vision_info
from TAM.tam import TAM

/home/mitchell/dev/hse/HSE_DL_4_2025/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
IMAGE_PATH = "image.jpg" 
USER_PROMPT = "Describe this image in detail."
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
model_name = "Qwen/Qwen2-VL-2B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name, torch_dtype=dtype, device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_name)

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.13s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


In [4]:
if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(f"Файл {IMAGE_PATH} не найден")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": IMAGE_PATH},
            {"type": "text", "text": USER_PROMPT},
        ],
    }
]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to(model.device)

In [5]:
generated = model.generate(
    **inputs,
    max_new_tokens=100,
    use_cache=True,
    output_hidden_states=True, 
    return_dict_in_generate=True
)

generated_ids = generated.sequences
processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

'system\nYou are a helpful assistant.\nuser\nDescribe this image in detail.\nassistant\nThe image depicts a dynamic scene of a basketball player in mid-air, performing a slam dunk. The player is wearing a camouflage-patterned jacket and blue shorts, with yellow socks. The basketball is in mid-air, slightly above the hoop, indicating the player is in the process of executing a dunk. The hoop is attached to a metal frame, and the background features a cityscape with tall buildings, including the iconic Empire State Building, suggesting the setting is in New York City. The sky is over'

In [6]:
logits = [model.lm_head(feats[-1]) for feats in generated.hidden_states]

# Специальные ID токенов для Qwen2-VL (взято из demo.py авторов)
# Они нужны, чтобы понять, где кончается картинка и начинается текст
special_ids = {
    'img_id': [151652, 151653],
    'prompt_id': [151653, [151645, 198, 151644, 77091]], 
    'answer_id': [[198, 151644, 77091, 198], -1]
}

# Вычисление размеров сетки токенов изображения
if 'image_grid_thw' in inputs:
    h_grid = inputs['image_grid_thw'][0, 1] // 2
    w_grid = inputs['image_grid_thw'][0, 2] // 2
    vision_shape = (h_grid, w_grid)
else:
    vision_shape = (16, 16)

vis_inputs = image_inputs[0]

In [7]:
raw_map_records = []
token_ids_list = generated_ids[0].cpu().tolist()

for i in range(len(logits)):
    save_path = os.path.join(OUTPUT_DIR, f"step_{i:03d}.jpg")    
    try:
        TAM(
            tokens=token_ids_list,
            vision_shape=vision_shape,
            logit_list=logits,
            special_ids=special_ids,
            vision_input=vis_inputs,
            processor=processor,
            save_fn=save_path,
            target_token=i,
            img_scores_list=raw_map_records,
            eval_only=False
        )
        print(f"Сохранен: {save_path}")
        
    except Exception as e:
        print(f"Ошибка на шаге {i}: {e}")
        import traceback
        traceback.print_exc()

print(f"Результаты в папке: {os.path.abspath(OUTPUT_DIR)}")

[ WARN:0@157.141] global loadsave.cpp:1063 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.


Сохранен: output/step_000.jpg
Сохранен: output/step_001.jpg
Сохранен: output/step_002.jpg
Сохранен: output/step_003.jpg
Сохранен: output/step_004.jpg
Сохранен: output/step_005.jpg
Сохранен: output/step_006.jpg
Сохранен: output/step_007.jpg
Сохранен: output/step_008.jpg
Сохранен: output/step_009.jpg
Сохранен: output/step_010.jpg
Сохранен: output/step_011.jpg
Сохранен: output/step_012.jpg
Сохранен: output/step_013.jpg
Сохранен: output/step_014.jpg
Сохранен: output/step_015.jpg
Сохранен: output/step_016.jpg
Сохранен: output/step_017.jpg
Сохранен: output/step_018.jpg
Сохранен: output/step_019.jpg
Сохранен: output/step_020.jpg
Сохранен: output/step_021.jpg
Сохранен: output/step_022.jpg
Сохранен: output/step_023.jpg
Сохранен: output/step_024.jpg
Сохранен: output/step_025.jpg
Сохранен: output/step_026.jpg
Сохранен: output/step_027.jpg
Сохранен: output/step_028.jpg
Сохранен: output/step_029.jpg
Сохранен: output/step_030.jpg
Сохранен: output/step_031.jpg
Сохранен: output/step_032.jpg
Сохранен: 

Проверил TAM на своей картинке в своем окружении. В целом, механизм действительно работает, можно видеть причинно-следственные связи.